# 仿真数据生成调试 Notebook

针对 **beaconless-ao-sim** 的数据生成流水线（`data/simulate.py`），
逐步交互式调试。每个单元对应算法 1 的一个阶段，可单独重跑、改参数、看中间量。

对应论文：DiComo 等，*Opt. Express* 33(15):31010 (2025)，DOI 10.1364/OE.561077。

**运行方式**：
```bash
cd /home/ws/code/beaconless-ao-sim
uv run jupyter notebook data/调试仿真数据生成.ipynb
```
（或 VS Code 直接打开；内核选用 uv 创建的 `.venv` 环境。）

**调试技巧**：
- 改 `cfg` 里任一参数后，重新运行「构建共享状态」单元即可让改动生效。
- 多数重计算（传播器 FFTW 初始化、Zernike 基、湍流屏生成）都缓存在 `shared` 里，
  跨单元复用，避免重复等待。
- 单个 `simulate_sample` 约 5–6 秒（含 10 层屏 + 10 个粗糙面散射成像）。

## 0. 环境准备与配置加载

导入依赖，加载 `config.yaml`，并设置绘图。所有后续单元都依赖这里的 `cfg`。

In [ ]:
import sys, os, time
import numpy as np
import yaml
import matplotlib
matplotlib.use('module://matplotlib_inline.backend_inline') if False else None
import matplotlib.pyplot as plt

# 确保从仓库根导入（notebook 在 data/ 子目录）
ROOT = os.path.dirname(os.path.dirname(os.path.abspath('__file__')))
ROOT = os.getcwd()  # 在仓库根运行 jupyter 时即为仓库根
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

# 加载配置
cfg = yaml.safe_load(open(os.path.join(ROOT, 'config.yaml'), encoding='utf-8'))

# 常用绘图风格
plt.rcParams['figure.dpi'] = 100
plt.rcParams['image.cmap'] = 'inferno'

print('仓库根:', ROOT)
print('N =', cfg['physical']['N'],  'box =', cfg['physical']['box_size'], 'm')
print('Cn2 =', cfg['physical']['cn2'],  'L =', cfg['physical']['L'], 'm')
print('Dscope =', cfg['physical']['Dscope'], 'm   rspot =', cfg['physical']['rspot'], 'm')
# 注: YAML 将 800e-9 解析为字符串, 这里强制转 float
print('wavelength =', float(cfg['physical']['wavelength'])*1e9, 'nm')
print('n_screens =', cfg['physical']['n_screens'],  'n_roughness =', cfg['physical']['n_roughness'])
print('beam_source =', cfg['physical']['beam_source'])
print('splits: train/test/eval =',
      cfg['data']['n_train'], cfg['data']['n_test'], cfg['data']['n_eval'])

## 1. 构建共享状态 `SharedSim`

`_get_shared(cfg)` 一次性构建整个进程复用、跨样本不变的重计算量：
- **Propagator**（FFT 分步传播器，FFTW 初始化约 1–3 s）
- **ZernikeBasis**（78 阶 Noll 基，伪逆约 1 s）
- 坐标网格 `X/Y/r2`、孔径掩膜 `pupil`、跟踪高斯权重 `G`
- 入瞳光束 `E0`、聚焦相位 `phi_focus`、真空目标面强度 `I_vac`
- 成像几何：`zR_APWS`、`f_obj`、三个测量平面距离 `plane_offsets`

**调试点**：检查 `zR_APWS` / `f_obj` 是否按公式 9–12 解析解正确
（800 nm、Cn2=8.13e-15、L=1 km 时 `zR≈642 m`、`f_obj≈1285 m`）。

In [ ]:
from data.simulate import (_get_shared, simulate_sample, _quantize,
                            _make_screens, _imaging, _fom_leg)

t0 = time.time()
shared = _get_shared(cfg)
print(f'共享状态构建耗时 {time.time()-t0:.1f} s')

print('N =', shared.N, ' dx =', round(shared.dx, 8), 'm')
print('lambda =', shared.lam, '  k =', round(shared.k, 1))
print('rspot =', shared.rspot, 'm   focal =', shared.focal, 'm')
print('zR_APWS =', round(shared.zR_APWS, 3), 'm   f_obj =', round(shared.f_obj, 3), 'm')
print('plane_offsets =', np.round(shared.plane_offsets, 3), 'm')
print('  (plane 0 = f_obj - zR, plane 1 = 焦平面, plane 2 = f_obj + zR)')
print('E0: shape', shared.E0.shape, 'max', round(float(shared.E0.max()),4))
print('pupil 内像素数 =', int(shared.pupil.sum()))
print('I_vac: shape', shared.I_vac.shape, 'max', round(float(shared.I_vac.max()),4))

### 1.1 可视化共享状态中的关键场

看入瞳光束 `|E0|^2`、聚焦相位 `phi_focus`、真空目标面 `I_vac`，确认光学链路设置正确。

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(13, 4))
ax[0].imshow(np.abs(shared.E0)**2, cmap='inferno')
ax[0].set_title('入瞳光束 |E0|^2 (rspot=%.2f m)' % shared.rspot)
ax[0].axis('off')
ax[1].imshow(np.angle(shared.E0) if False else shared.phi_focus, cmap='twilight',
             vmin=-np.pi, vmax=np.pi)
ax[1].set_title('聚焦相位 phi_focus (rad)')
ax[1].axis('off')
ax[2].imshow(shared.I_vac, cmap='inferno',
             vmin=0, vmax=np.percentile(shared.I_vac, 99.9))
ax[2].set_title('真空目标面 I_vac (无湍流)')
ax[2].axis('off')
plt.tight_layout()
plt.show()

## 2. 湍流相位屏 `_make_screens(seed, cfg, shared)`

生成 `n_screens` 层确定性相位屏（rad）。
- `beam_source: oopao` → 从内嵌 OOPAO 大气中抽取 von-Karman 屏，每层缩放到目标
  每 slab r0（`r0_slab = r0_path · n^(3/5)`）后裁剪到 `N×N`。
- `beam_source: soapy/aotools` → 用 `ft_sh_phase_screen` 逐层生成，屏 i 用种子 `seed+i`。

**调试点**：检查每层 OPD 标准差是否落在合理范围；改 `cfg['physical']['cn2']` 或
`beam_source` 后对比屏幕强度。

In [ ]:
SEED = int(cfg['data']['master_seed'])   # 第 0 个训练样本的种子
t0 = time.time()
screens = _make_screens(SEED, cfg, shared)
print(f'屏幕生成耗时 {time.time()-t0:.1f} s   shape = {screens.shape} (n_screens, N, N)')

fig, ax = plt.subplots(1, min(4, screens.shape[0]), figsize=(13, 3))
for i, a in enumerate(ax if isinstance(ax, np.ndarray) else [ax]):
    a.imshow(screens[i], cmap='twilight', vmin=-np.pi, vmax=np.pi)
    a.set_title(f'屏 {i}  std={screens[i].std():.3f} rad')
    a.axis('off')
plt.tight_layout(); plt.show()

print('每层 OPD 标准差 (rad):', np.round(screens.std(axis=(1,2)), 4))

## 3. 算法 1 导引信标反向传播

发射衍射极限高斯信标，传播到目标后**反向传播回瞳孔**，并做解析抛物面离焦移除，
得到共轭信标相位 `phi_conj`（无信标 AO 的波前估计）。

这里直接调用 `simulate_sample`，它会返回所有中间量，便于逐步检查。
（`simulate_sample` 内部会重建屏幕、做信标、跟踪、Zernike 投影、FOM 与成像。）

In [ ]:
t0 = time.time()
sample = simulate_sample(SEED, cfg, shared=shared)
print(f'simulate_sample 耗时 {time.time()-t0:.1f} s   seed = {sample.seed}')

print('各相位图 shape:', {k: v.shape for k, v in {
    'phi_conj': sample.phase_conj, 'phi_track': sample.phase_track,
    'phi_beacon': sample.phase_beacon, 'phi_z78': sample.phase_z78}.items()})
print('track_slopes [ax, ay] =', np.round(sample.track_slopes, 6))
print('labels (78 阶 Zernike, rad) 范数 =', round(float(np.linalg.norm(sample.labels)), 4))
print('FOM:  noao=%.4f  track=%.4f  beacon=%.4f  z78=%.4f' % (
    sample.fom_noao, sample.fom_track, sample.fom_beacon, sample.fom_z78))

### 3.1 信标相位与修正相位可视化

对比 `phi_conj`（共轭信标）、`phi_beacon = phi_conj - phi_track`（残余相位）、
`phi_z78`（78 阶 Zernike 重构）。

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(15, 3.5))
for a, (name, ph) in zip(ax, {
    'phi_conj (共轭信标)': sample.phase_conj,
    'phi_track (跟踪)':   sample.phase_track,
    'phi_beacon (残余)':  sample.phase_beacon,
    'phi_z78 (78阶重构)': sample.phase_z78}.items()):
    a.imshow(ph, cmap='twilight', vmin=-np.pi, vmax=np.pi)
    a.set_title(name + f'\nstd={ph.std():.3f}')
    a.axis('off')
plt.tight_layout(); plt.show()

## 4. 多平面成像 `_imaging`（核心调试对象）

这是之前出现「焦平面中心暗 / 边缘亮」伪影、并已修复的地方。流程：
1. 跟踪条件目标面场 `E_obj_track` → 强度 `I_obj_track`；
2. 对 `n_roughness` 个独立粗糙面散射；
3. 反向传播回望远镜（**乘以 `pupil` 吸收边界**，移除孔径外场，防止 FFT 卷绕）；
4. 共轭准直 + 物镜聚焦；
5. 传播到 3 个测量平面，**逐平面强度再乘 `pupil`**，并按 realization 非相干平均。

**调试点**：逐平面检查焦平面（plane 1）中心/边缘强度比，确认边缘卷绕已消除。

In [ ]:
images, I_obj_track = _imaging(SEED, cfg, shared, screens, sample.phase_track)
print('images shape:', images.shape, '  dtype:', images.dtype)
print('I_obj_track max =', round(float(I_obj_track.max()), 4))

# 逐平面中心/边缘强度比（焦平面应中心 >> 边缘）
N = shared.N
rr = np.sqrt((np.arange(N)[:,None]-N//2)**2 + (np.arange(N)[None,:]-N//2)**2).astype(int)
for p in range(3):
    pl = images[p]
    c = pl[N//2-16:N//2+16, N//2-16:N//2+16].mean()
    e = pl[rr > int(N*0.40)].mean()
    print(f'  plane {p}: 中心={c:.3f}  边缘(r>{int(N*0.40)})={e:.3f}  中心/边缘={c/max(e,1e-9):.1f}x')

### 4.1 成像结果蒙太奇（3 平面 × 多 realization 平均）

自动对比度显示。焦平面（中间列）应为**亮核 + 暗晕**，边缘无点亮。

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(12, 4))
for p, a in enumerate(ax):
    lo, hi = np.percentile(images[p], [5, 99.9])
    a.imshow(np.clip((images[p]-lo)/(hi-lo), 0, 1), cmap='inferno')
    a.set_title(f'plane {p}  (max={images[p].max():.3f})')
    a.axis('off')
plt.suptitle('多平面成像（修正后：吸收边界 + 非相干平均）')
plt.tight_layout(); plt.show()

## 5. 逐图像归一化 + 12-bit 量化 `_quantize`

图 2：「图像分别进行了归一化处理」。每张图按自身最大值缩放到 12-bit 满量程（2047），
焦平面（能量集中）因此达到满深度，而不会被数据集/逐平面全局最大值压暗。
`scale_p` 仍保留在 HDF5 中用于 schema 兼容，但不再用于量化。

In [ ]:
# 逐平面 raw 最大值（schema 兼容用）
scale_p = images.max(axis=(1,2))
q = _quantize(images, scale_p)
print('量化后 shape:', q.shape, '  dtype:', q.dtype)
print('每平面量化 max =', [int(q[p].max()) for p in range(3)], ' (应均为 2047)')
print('scale_p (raw 每平面 max) =', np.round(scale_p, 4))

fig, ax = plt.subplots(1, 3, figsize=(12, 4))
for p, a in enumerate(ax):
    a.imshow(q[p], cmap='inferno', vmin=0, vmax=2047)
    a.set_title(f'plane {p}  量化 (max={int(q[p].max())})')
    a.axis('off')
plt.suptitle('12-bit 逐图像归一化量化结果')
plt.tight_layout(); plt.show()

## 6. FOM 各分支（nPIB 桶积分）

用 `_fom_leg` 单独重算各修正分支的 FOM，理解 gain / eta 的来源：
- `noao`：无校正（仅聚焦）
- `track`：仅倾斜跟踪
- `beacon`：信标共轭（理想相位共轭上界）
- `z78`：78 阶 Zernike 重构上界
- `ml`：给定预测系数（可传入 CNN 预测）

**调试点**：`gain = FOM_ML / FOM_track`，`eta = (FOM_ML-FOM_track)/(FOM_z78-FOM_track)`。

In [ ]:
# 重算各分支（phi_focus + 修正相位）
print('重算 FOM 各分支：')
for name, phi in {'noao': shared.phi_focus,
                  'track': shared.phi_focus + sample.phase_track,
                  'beacon': shared.phi_focus + sample.phase_track + sample.phase_beacon,
                  'z78': shared.phi_focus + sample.phase_track + sample.phase_z78}.items():
    f = _fom_leg(shared, screens, phi)
    print(f'  {name:7s} FOM = {f:.4f}')

# 用一个「预测系数」演示 ML 分支（这里用真实标签模拟完美预测）
fom_ml = _fom_leg(shared, screens,
                  shared.phi_focus + sample.phase_track + sample.phase_z78)  # 示例
print(f'  ml     FOM = {fom_ml:.4f}  (示例：用 z78 相位作为预测)')

track_f = sample.fom_track; z78_f = sample.fom_z78
print(f'若 FOM_ML={fom_ml:.4f}: gain={fom_ml/track_f:.3f}  eta={(fom_ml-track_f)/(z78_f-track_f):.3f}')

## 7. 批量生成与 HDF5 数据集

`generate_dataset(cfg)` 跑完整两趟流水线：
- **Pass 1**：在训练集上算逐平面强度最大值 + 逐模式标签 mean/std（公式 13–14）；
- **Pass 2**：量化并把所有样本流式写入 HDF5（分块写，不在内存累积 raw 图）。

**调试点**：把 `cfg['data']` 的 `n_train/n_test/n_eval` 调小（如 8/4/8）、
`workers` 调小（如 4）、`h5_path` 改到 `/tmp`，快速验证整条流水线后再跑全量。
完整 `data/generate_h5` CLI 等价于：
```bash
uv run python -m data.generate_h5 --config config.yaml
```

In [ ]:
# 快速小批量验证（不写大文件，先跑通逻辑）
import copy
cfg_small = copy.deepcopy(cfg)
cfg_small['data'].update({'n_train': 8, 'n_test': 4, 'n_eval': 8,
                          'workers': 4,
                          'h5_path': '/tmp/opencode/demo_debug.h5'})

from data.simulate import generate_dataset
t0 = time.time()
h5_path = generate_dataset(cfg_small)
print(f'小批量数据集生成耗时 {time.time()-t0:.1f} s -> {h5_path}')

## 8. 读回 HDF5 验证

检查写出的数据集：图像 shape/dtype、逐平面 max（应 2047）、标签、FOM 中位数。

In [ ]:
import h5py
with h5py.File(h5_path, 'r') as f:
    print('HDF5 数据集 keys:', list(f.keys()))
    imgs = f['images']
    print('images:', imgs.shape, imgs.dtype, '  全局 max =', int(imgs[:].max()))
    # images 形状: (n_total, 3, N, N) -> 逐平面检查
    if imgs.ndim == 4 and imgs.shape[1] == 3:
        for p in range(3):
            print(f'  plane {p} max = {int(imgs[:,:,p,:].max())}  (应 2047)')
    elif imgs.ndim == 5:
        for p in range(3):
            print(f'  plane {p} max = {int(imgs[:,:,:,p,:].max())}  (应 2047)')
    lab = f['labels']
    print('labels:', lab.shape, lab.dtype)
    for k in ['fom_noao','fom_track','fom_beacon','fom_z78']:
        if k in f:
            print(f'  {k} median = {float(np.median(f[k][:])):.4f}')
    if 'scale_p' in f:
        print('scale_p (逐平面 raw max) =', np.round(f['scale_p'][:], 4))

## 9. 参数敏感性扫描（可选）

改 `cfg` 的物理参数（如 Cn2、n_roughness、Dscope）后重新构建 `shared` 并对比 FOM / 图像。
下面示例扫描 Cn2，观察湍流强度对 FOM 的影响。

> 注意：改 `N/box/wavelength/Dscope/rspot/focal/L/cn2/...` 等会改变 `_cfg_key`，
> 触发重建 `shared`（约 4 s）。改 `n_roughness` 不重建 shared，只影响成像循环。

In [ ]:
import copy
results = []
for cn2 in [4e-15, 8.13e-15, 1.6e-14]:
    c = copy.deepcopy(cfg); c['physical']['cn2'] = cn2
    sh = _get_shared(c)                       # cn2 改变 -> 重建 shared
    scr = _make_screens(SEED, c, sh)
    s = simulate_sample(SEED, c, shared=sh)
    results.append((cn2, s.fom_noao, s.fom_track, s.fom_beacon, s.fom_z78))
    print(f'Cn2={cn2:.1e}: noao={s.fom_noao:.3f} track={s.fom_track:.3f} '
          f'beacon={s.fom_beacon:.3f} z78={s.fom_z78:.3f}')

arr = np.array(results)
plt.figure(figsize=(7,4))
plt.plot(arr[:,0], arr[:,1], 'o-', label='noao')
plt.plot(arr[:,0], arr[:,2], 's-', label='track')
plt.plot(arr[:,0], arr[:,3], '^-', label='beacon')
plt.plot(arr[:,0], arr[:,4], 'd-', label='z78')
plt.xscale('log'); plt.xlabel('Cn2 [m^-2/3]'); plt.ylabel('FOM')
plt.title('FOM vs Cn2（湍流强度敏感性）'); plt.legend(); plt.grid(True)
plt.tight_layout(); plt.show()